# day 4

In [1]:
# Importações
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse


In [2]:
# Carregar dados do MovieLens
ratings = pd.read_csv("u.data", sep="\t", names=['userId', 'movieId', 'rating', 'timestamp'])
movies = pd.read_csv("u.item", sep='|', encoding='latin-1', names=[
    'movieId', 'title', 'release_date', 'video_release_date', 'IMDb_URL',
    'unknown','Action','Adventure','Animation',"Children's",'Comedy','Crime',
    'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
    'Romance','Sci-Fi','Thriller','War','Western'
], usecols=range(24))  # Só precisamos das primeiras 24 colunas

### Preparar o dataset para o Surprise

In [3]:

reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

### Divisão treino/teste

In [4]:

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

### Treinar modelo SVD (Collaborative Filtering)

In [5]:

model = SVD()
model.fit(trainset)

# Fazer previsões no conjunto de teste

In [6]:

predictions = model.test(testset)
rmse(predictions)


RMSE: 0.9348


0.9348218924059858


### Função para recomendar filmes

In [7]:

def recomendar_filmes(usuario_id, top_n=5):
    # Todos os filmes
    todos_filmes = movies['movieId'].unique()
    
    # Filmes que o usuário já avaliou
    avaliados = ratings[ratings.userId==usuario_id]['movieId'].tolist()
    
    # Filmes não avaliados
    nao_avaliados = [f for f in todos_filmes if f not in avaliados]
    
    # Prever nota para filmes não avaliados
    notas_preditas = [ (f, model.predict(usuario_id, f).est) for f in nao_avaliados ]
    
    # Ordenar e pegar top_n
    top_filmes = sorted(notas_preditas, key=lambda x: x[1], reverse=True)[:top_n]
    
    # Retornar nomes dos filmes
    top_filmes_ids = [f[0] for f in top_filmes]
    return movies[movies.movieId.isin(top_filmes_ids)][['title']]

# Exemplo de recomendação para usuário 1
recomendar_filmes(1, top_n=5)


,title
356,One Flew Over the Cuckoo's Nest (1975)
407,"Close Shave, A (1995)"
426,To Kill a Mockingbird (1962)
511,Wings of Desire (1987)
1239,Ghost in the Shell (Kokaku kidotai) (1995)
